# Phase 3 — Source-Held-Out Probes

**The question.** Does the model read clauses, or does it recognise datasets? The fused
corpus stitches three annotation projects together, each with its own drafting register,
clause segmentation and label vocabulary. A model that has learned "this looks like a
CLAUDETTE row, and CLAUDETTE rows about arbitration are usually harmful" would score well
on a random split while having learned very little about arbitration.

The existing `notebooks/model_finetuning/lawgic_classifier_probe.ipynb` already showed
that source identity is *linearly decodable* from the fine-tuned encoder. That is
necessary but not sufficient evidence: an encoder can carry source information without the
heads depending on it. This notebook tests the stronger claim directly — **remove a source
from training entirely, then evaluate only on that source's rows.**

## Two probes, and why not three

| Probe | Held out | Corpus rows carrying that source |
| --- | --- | --- |
| A | CLAUDETTE | 3,182 wide rows (3,721 long-format annotation rows) |
| B | 100 ToS | 1,460 wide rows (2,048 long-format annotation rows) |
| — | ~~ToS;DR~~ | **deliberately not run** |

The row counts differ between the long and wide formats because the wide corpus is one row
per unique clause: a clause annotated with several topics by the same source collapses into
one row with several active topic cells. The holdout operates on wide rows, so those are
the numbers reported.

**Why there is no ToS;DR holdout.** ToS;DR supplies the majority of the corpus rows — the majority of the corpus and training rows once the split is applied. Removing it would
leave very few training clauses. A score collapse under that condition is uninterpretable:
it would be perfectly consistent with "the model only recognised ToS;DR" *and* with "no
model learns 44-way multi-label legal topic detection from 2,500 examples". The probe
would be confounded with data starvation and would answer neither question. The two
smaller sources can be removed while leaving the training regime broadly intact, which is
what makes their results readable.

## Protocol

Legal-BERT, seed 42, dual-head, identical to Phase 2 in every other respect — same
persisted seed-42 split, same hyperparameters, same losses, same early stopping, same
degenerate-model assertion. The holdout removes the source's rows from **train and
validation** and restricts the **test** set to exactly those rows.


## Masking: score only what the held-out source actually supervised

This is the part that decides whether the probe means anything, and the first pass got the
direction right and the width wrong.

The supervision mask is source-aware: a row annotated by CLAUDETTE has observed cells only
for the topics CLAUDETTE's label vocabulary covers; every other cell is *unknown* and
contributes zero loss. If a held-out CLAUDETTE row is scored across all predicted topics, most of
the score comes from cells CLAUDETTE never labelled — the model is being graded against
the shape of the mask, not against comprehension of the clause. So the surface must be
restricted.

**But it must not be restricted to the cells the source marked positive.**
`source_supervision_mask()` in `scripts/lawgic_train_matrix.py` rebuilds the mask from
`native_annotations`, and `native_annotations` records only *asserted* annotations — that
is, only positives. The resulting surface therefore contains **zero observed negatives**
(CLAUDETTE: 375 cells, 375 positive; 100 ToS: 197 cells, 197 positive). False positives
become structurally impossible, micro-precision comes out at exactly 1.000 in every
condition including in-distribution, and per topic F1 collapses to `2r/(1+r)` — a monotone
function of recall alone. A model predicting every topic present for every clause would
score a perfect 1.000 on that surface. It is the positive-only-corpus degeneracy this
project already fixed once at the corpus level, reappearing one layer up in the evaluation.

So two surfaces are reported below:

| Surface | Mask | Observed negatives | What it measures |
| --- | --- | --- | --- |
| **asserted** | `tm.source_supervision_mask()` | none | recall on the topics the source marked; the as-first-published figures |
| **corpus** | the row's own `label_mask` | yes | precision *and* recall; the honest F1 |

The corpus mask is the fusion pipeline's own source-aware mask, built by applying the
mapping policy, so for a row annotated by one source it *is* that source's supervision.
`source_mappings` in `lawgic_topics*.json` cannot be used instead: it lists every topic a
source's raw labels touch, including the fine subtypes the mapping policy deliberately
refuses to assert (CLAUDETTE's `ltd` maps to `limitation_of_liability` alone, not to
`liability_cap` and `warranty_disclaimer`), so masking on it would manufacture false
negatives — the exact error the mapping layer exists to avoid.

Rows carrying more than one source would mix their masks, so `PURE_SOURCE_ONLY` restricts
the corpus surface to rows annotated by the held-out source alone. That is 96% of them
(cross-source overlap is 0.4% of the corpus).

The in-distribution baseline is computed on **exactly the same rows and the same cell
mask**, from the Phase 2 legal-bert/seed-42 run's stored logits. Without that restriction
the "retained ratio" would compare two different denominators and would be meaningless.


In [ ]:
import os
import sys
from pathlib import Path

# ── Corpus version: set BEFORE importing lawgic_eval_core ─────────────────────
os.environ["LAWGIC_CORPUS_VERSION"] = "v2"


def find_project_root(start: Path) -> Path:
    for sentinel in [
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide_v2.csv",
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv",
    ]:
        for candidate in (start, *start.parents):
            if (candidate / sentinel).exists():
                return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS

HOLDOUT_SOURCES = ["claudette", "100_tos"]
BASELINE_RUN = tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual").run_id

for source in HOLDOUT_SOURCES + ["tos_dr"]:
    corpus_rows = int(tm.source_row_mask(corpus, source).sum())
    train_rows = int(tm.source_row_mask(frames["train"], source).sum())
    test_rows = int(tm.source_row_mask(frames["test"], source).sum())
    print(f"{source:>10}: corpus {corpus_rows:>6,} | train {train_rows:>6,} "
          f"({train_rows / len(frames['train']):.1%}) | test {test_rows:>5,}")

print(f"\nCorpus version: {core._CORPUS_VERSION}")
print(f"Topics: {core.NUM_LAWGIC_TOPICS}")
print(f"Runs dir: {tm.RUNS_DIR}")
print(f"Phase 2 baseline run required: {BASELINE_RUN}")


## Prerequisites

1. **Phase 2 must have run.** This notebook reads
   `generated_files/lawgic_taxonomy/runs/legal-bert-base-uncased__seed42__dual/test_logits.npz`
   for the in-distribution baseline. Without it there is nothing to compare against.
2. **No GPU needed.** Both probes are already trained and their test logits are persisted, so
   every cell below re-scores stored arrays and runs on a laptop in seconds. The training call
   is commented out; uncomment it only if a probe directory is missing.
3. No downloads or credentials.

In [ ]:
baseline_path = tm.RUNS_DIR / BASELINE_RUN / "test_logits.npz"
if not baseline_path.exists():
    raise FileNotFoundError(
        f"{baseline_path} missing. Run 02_multiseed_encoder_runs.ipynb (at least the "
        f"legal-bert/seed42/dual config) before this notebook."
    )
print(f"Baseline logits found: {baseline_path}")

## Load the two probes

**Both probes have already been trained** (`legal-bert-base-uncased__seed42__dual__holdout-claudette`
and `…__holdout-100_tos`, 39.3 and 39.0 min, 14 and 13 epochs, both early-stopped inside the
20-epoch budget). Everything after this cell re-scores their **persisted logits**, so the rest of
the notebook runs on a laptop in seconds with no GPU.

The training call is therefore commented out. Uncomment it only if a probe directory is missing
or if the protocol changes.

In [ ]:
probe_configs = [
    tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual", holdout_source=source)
    for source in HOLDOUT_SOURCES
]

probe_records = []
for config in probe_configs:
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists():
        print(f"loaded {config.run_id}")
        probe_records.append(json.loads(target.read_text()))
        continue

    # ── Retraining is disabled: both probes are already on disk. ────────────
    # Uncomment to (re)train a missing probe. ~40 min per run on a CUDA GPU.
    #
    # print(f"running {config.run_id} ...")
    # probe_records.append(tm.run_config(config))
    raise FileNotFoundError(
        f"{target} missing. Uncomment the tm.run_config call above to train this probe."
    )

display(pd.DataFrame(probe_records)[
    ["run_id", "train_rows", "val_rows", "test_rows", "wall_seconds", "epochs_run",
     *core.HEADLINE_METRICS]
])
print("\nNote: the metrics above are on the *asserted* surface (positives only) — that is what "
      "run_config computed at training time. The corrected figures follow below.")

## In-distribution baseline on the same rows and the same cells

The Phase 2 run scored the full test split. Here it is re-scored on the subset of
test rows belonging to the held-out source, under both candidate masks — the identical
evaluation surface the probe faces, so the only difference between the two conditions is
whether the source appeared in training.

Both conditions are recomputed from persisted logits by the same code path, rather than
reading the probe's `metrics.json`, so the two masks can be applied consistently and the
`risk_majority_floor` column can be computed on the same rows. A metric without its floor
is uninterpretable, and the floor moves with the subset: predicting *neutral* everywhere
scores 0.744 on CLAUDETTE's test rows and 0.610 on 100 ToS's, against 0.469 on the full
test split.


In [ ]:
SURFACES = ("corpus", "asserted")   # first one is the headline
CONDITIONS = ("in-distribution (Phase 2)", "held-out (Phase 3)")
IN_DIST, HELD_OUT = CONDITIONS

baseline = np.load(baseline_path)
base_position = {int(r): i for i, r in enumerate(baseline["row_id"])}
test_frame = frames["test"]


def micro_pr(topic_logits, labels, masks):
    """Micro precision / recall over observed cells. Catches over-prediction, which the
    asserted surface cannot: with no observed negatives, fp is 0 by construction."""
    pred = core.logits_to_predictions(topic_logits).astype(bool)
    lab, obs = labels.astype(bool), masks.astype(bool)
    tp = int((pred & lab & obs).sum())
    fp = int((pred & ~lab & obs).sum())
    fn = int((~pred & lab & obs).sum())
    return {
        "micro_precision": tp / (tp + fp) if tp + fp else 0.0,
        "micro_recall": tp / (tp + fn) if tp + fn else 0.0,
        "false_positives": fp,
    }


def risk_majority_floor(arrays):
    """Accuracy of always predicting the most frequent risk class on these rows."""
    valid = arrays["harm_masks"].astype(bool)
    counts = np.bincount(arrays["harm_labels"][valid].astype(int), minlength=core.NUM_HARM_CLASSES)
    return float(counts.max() / counts.sum())


def probe_surface(source, pure_source_only=False):
    """Aligned logits for both conditions plus both candidate masks, on one probe's rows."""
    probe = np.load(tm.RUNS_DIR / f"{BASELINE_RUN}__holdout-{source}" / "test_logits.npz")
    probe_position = {int(r): i for i, r in enumerate(probe["row_id"])}

    rows = test_frame[tm.source_row_mask(test_frame, source)].copy()
    if pure_source_only:
        rows = rows[rows["sources"].map(lambda s: list(s) == [source])].copy()

    row_ids = [int(r) for r in rows["row_id"]]
    base_idx = np.array([base_position[i] for i in row_ids])
    probe_idx = np.array([probe_position[i] for i in row_ids])

    arrays = core.label_arrays(rows)
    return {
        "source": source,
        "arrays": arrays,
        # "corpus": the fusion pipeline's own source-aware mask, which carries observed
        # negatives. "asserted": only the cells this source marked positive.
        "masks": {"corpus": arrays["label_masks"],
                  "asserted": tm.source_supervision_mask(rows, source)},
        "logits": {IN_DIST: (baseline["topic_logits"][base_idx], baseline["harm_logits"][base_idx]),
                   HELD_OUT: (probe["topic_logits"][probe_idx], probe["harm_logits"][probe_idx])},
        "n_rows": len(rows),
    }


probes = {source: probe_surface(source) for source in HOLDOUT_SOURCES}

records = []
for source, probe in probes.items():
    arrays = probe["arrays"]
    floor = risk_majority_floor(arrays)
    for surface in SURFACES:
        mask = probe["masks"][surface]
        scoped = {**arrays, "label_masks": mask}
        observed = int(mask.sum())
        positives = int((arrays["labels"] * mask).sum())
        for condition, (topic_logits, harm_logits) in probe["logits"].items():
            records.append({
                "source": source,
                "surface": surface,
                "condition": condition,
                "rows": probe["n_rows"],
                "observed_cells": observed,
                "positive_cells": positives,
                "negative_cells": observed - positives,
                **core.all_metrics(topic_logits, harm_logits, scoped),
                **micro_pr(topic_logits, arrays["labels"], mask),
                "risk_majority_floor": floor,
            })

comparison = pd.DataFrame(records).drop(columns=["topic_observed_positions", "risk_valid_rows"])
display(comparison.set_index(["source", "surface", "condition"]).round(4))

# The defect, stated as an assertion rather than a claim.
asserted = comparison[comparison["surface"] == "asserted"]
assert (asserted["negative_cells"] == 0).all(), "asserted surface unexpectedly has negatives"
assert (asserted["false_positives"] == 0).all()
print("\nasserted surface: 0 observed negatives and 0 false positives in every condition — "
      "its F1 is a recall proxy, precision is free.")

# Sensitivity check: does dropping the few multi-source rows move anything?
for source in HOLDOUT_SOURCES:
    pure = probe_surface(source, pure_source_only=True)
    scoped = {**pure["arrays"], "label_masks": pure["masks"]["corpus"]}
    pure_held = core.all_metrics(*pure["logits"][HELD_OUT], scoped)
    full = comparison[(comparison["source"] == source)
                      & (comparison["surface"] == "corpus")
                      & (comparison["condition"] == HELD_OUT)].iloc[0]
    print(f"{source:>10}: corpus-surface held-out micro-F1 "
          f"{full['topic_micro_f1']:.4f} on {full['rows']} rows | "
          f"{pure_held['topic_micro_f1']:.4f} on {pure['n_rows']} single-source rows")

## Retained-performance ratio, with a paired bootstrap on the difference

`held-out / in-distribution`, per metric. Read it as: **what fraction of its ability does the
model keep when it has never seen a single clause from this source?**

- **near 1.0** — the model generalises across annotation projects; performance is not an
  artifact of source recognition.
- **materially below 1.0** — part of the headline score depends on having seen that source's
  register during training. That is a real limitation of the fused-corpus design, not
  necessarily a modelling failure.

A ratio on its own is not evidence, so two things are attached to it. The **observed-cell
count** is reported so the ratio is never read without its denominator. And the difference
between the two conditions is **paired-bootstrapped over the same clause resamples**, because
both conditions scored identical rows in identical order — if that interval contains zero, the
drop is indistinguishable from sampling noise. It does not contain zero here.

Metrics are reported on both surfaces. Prefer the `corpus` rows: on the `asserted` surface the
precision columns are 1.000 by construction, so its `topic_macro_f1` is a recall proxy.

In [ ]:
RATIO_METRICS = (*core.HEADLINE_METRICS, "micro_precision", "micro_recall")
N_RESAMPLES = 1000


def delta_ci(probe, mask, metric):
    """Paired bootstrap 95% CI on (held-out minus in-distribution) for one metric.

    Both conditions are scored on the *same* resampled clause indices, so clause
    difficulty cancels — the interval is over the difference, not over two scores.
    """
    arrays = {**probe["arrays"], "label_masks": mask}
    in_topic, in_harm = probe["logits"][IN_DIST]
    held_topic, held_harm = probe["logits"][HELD_OUT]

    def scorer(logits):
        topic_logits, harm_logits = logits

        def score(idx):
            values = core.all_metrics(topic_logits, harm_logits, arrays, idx)
            if metric in values:
                return values[metric]
            return micro_pr(topic_logits[idx], arrays["labels"][idx], mask[idx])[metric]

        return score

    held_score, in_score = scorer((held_topic, held_harm)), scorer((in_topic, in_harm))
    return core.paired_bootstrap_delta(
        lambda idx: held_score(idx) - in_score(idx),
        np.arange(probe["n_rows"]),
        n_resamples=N_RESAMPLES,
    )


ratio_rows = []
for source, probe in probes.items():
    for surface in SURFACES:
        mask = probe["masks"][surface]
        scored = {
            condition: {
                **core.all_metrics(*logits, {**probe["arrays"], "label_masks": mask}),
                **micro_pr(logits[0], probe["arrays"]["labels"], mask),
            }
            for condition, logits in probe["logits"].items()
        }
        for metric in RATIO_METRICS:
            in_value, held_value = scored[IN_DIST][metric], scored[HELD_OUT][metric]
            test = delta_ci(probe, mask, metric)
            ratio_rows.append({
                "Source": source,
                "Surface": surface,
                "Metric": metric,
                "In-distribution": in_value,
                "Held-out": held_value,
                "Retained ratio": held_value / in_value if in_value else float("nan"),
                "Delta": test["delta"],
                "CI low": test["ci_low"],
                "CI high": test["ci_high"],
                "Test rows": probe["n_rows"],
                "Observed cells": int(mask.sum()),
                "Observed negatives": int(mask.sum() - (probe["arrays"]["labels"] * mask).sum()),
            })

probe_table = pd.DataFrame(ratio_rows)
for surface in SURFACES:
    print(f"\n=== {surface} surface ===")
    display(probe_table[probe_table["Surface"] == surface].drop(columns="Surface").round(4))

core.write_outputs(
    probe_table[probe_table["Surface"] == "corpus"].drop(columns="Surface"),
    "phase3_source_holdout",
    caption=(
        "Source-held-out probes. Each probe retrains Legal-BERT (seed 42, dual-head, "
        "identical protocol) with one source removed from train and validation, then "
        "evaluates only on that source's test rows under that source's own supervision "
        "mask. The in-distribution column is the Phase 2 legal-bert/seed-42 run scored on "
        "the identical rows and cells, so the columns differ only in training experience. "
        "Confidence intervals are percentile bootstrap over 1{,}000 clause-level resamples "
        "of the paired difference. No ToS;DR probe is reported: it would remove ~83\\% of "
        "training rows, confounding source recognition with data starvation."
    ),
    label="tab:source-holdout",
)

core.write_outputs(
    probe_table[probe_table["Surface"] == "asserted"].drop(columns="Surface"),
    "phase3_source_holdout_asserted",
    caption=(
        "Source-held-out probes scored on the asserted-cells surface, which contains only "
        "the topic cells the held-out source marked positive and therefore no observed "
        "negatives. Precision is 1.000 by construction, so these F1 figures are recall "
        "proxies; they are reported for continuity with the run-time metrics in "
        "\\texttt{metrics.json}. Table \\ref{tab:source-holdout} is the headline result."
    ),
    label="tab:source-holdout-asserted",
)

### Per-topic detail, joined against the supervision that survives the holdout

Which topics survive and which collapse is more informative than the aggregate — but the
aggregate cannot be decomposed without one extra column, and without that column the probe
cannot do its job.

A topic can score zero in a probe for two completely different reasons:

- **the holdout removed its supervision.** `liability_cap`, `severability`, `service_changes`,
  `price_changes` and `limitation_period` are supplied by 100 ToS *alone*, and
  `limitation_of_liability` draws 89% of its training positives from CLAUDETTE. In the probe
  condition these topics have little or no supervision left, so a zero measures
  **unlearnability**, not a failure to read. They must be excluded from any claim about source
  recognition — this is the per-topic reappearance of the same data-starvation confound that
  makes a ToS;DR probe uninterpretable.
- **the register did not transfer.** A topic that keeps 92–99% of its supervision and still
  scores zero is the real evidence of source dependence: the model learned what the training
  source's *phrasing* of the topic looks like rather than what the contractual mechanism is.

`train_positives_lost` and `train_positives_remaining` below separate the two. Read the
`observed` column first — topics with a handful of observed cells swing wildly and are
anecdote, not measurement.

In [ ]:
TOPIC_IDS, _, _ = core.load_taxonomy()
AVERAGE_ROWS = ["macro avg", "weighted avg"]

# Training positives per topic per source, from the long-format evidence on train rows.
train_positives = {}
for annotations in frames["train"]["native_annotations"]:
    for annotation in annotations or []:
        key = (annotation.get("lawgic_topic_id"), annotation.get("source_dataset"))
        train_positives[key] = train_positives.get(key, 0) + 1
supervision = (
    pd.DataFrame([{"topic_id": t, "source": s, "n": n} for (t, s), n in train_positives.items()])
    .pivot_table(index="topic_id", columns="source", values="n", fill_value=0)
    .astype(int)
)

per_topic_tables = {}
for source in HOLDOUT_SOURCES:
    # Single-source rows only here. On a co-annotated row the corpus mask is a union, so
    # the other source's cells would enter this table as one- or two-clause topics and
    # read as findings about the held-out source. The aggregates above are unaffected by
    # the restriction (see the sensitivity check in the previous cell).
    probe = probe_surface(source, pure_source_only=True)
    arrays = {**probe["arrays"], "label_masks": probe["masks"]["corpus"]}
    table = core.per_topic_table(probe["logits"][HELD_OUT][0], arrays, TOPIC_IDS)
    in_dist = core.per_topic_table(probe["logits"][IN_DIST][0], arrays, TOPIC_IDS)
    table = table.merge(
        in_dist[["topic_id", "f1"]].rename(columns={"f1": "f1_in_distribution"}),
        on="topic_id", how="left",
    )

    lost = supervision.get(source, pd.Series(dtype=int))
    remaining = supervision.drop(columns=[source], errors="ignore").sum(axis=1)
    table["train_positives_lost"] = table["topic_id"].map(lost).fillna(0).astype(int)
    table["train_positives_remaining"] = table["topic_id"].map(remaining).fillna(0).astype(int)
    table["share_lost"] = table["train_positives_lost"] / (
        table["train_positives_lost"] + table["train_positives_remaining"]
    ).replace(0, np.nan)
    per_topic_tables[source] = table

    # Topics with no positive in this subset carry no information about the holdout.
    scored = table[~table["topic_id"].isin(AVERAGE_ROWS) & (table["support"] > 0)]

    print(f"\n=== {source}: {probe['n_rows']} single-source rows, topics with at least one positive ===")
    print(scored.sort_values("f1")[
        ["topic_id", "f1", "f1_in_distribution", "precision", "recall", "support", "observed",
         "train_positives_lost", "train_positives_remaining", "share_lost"]
    ].round(4).to_string(index=False))

    starved = scored[scored["train_positives_remaining"] == 0]
    survived = scored[scored["train_positives_remaining"] > 0]
    print(f"\n  macro-F1 over all {len(scored)} scored topics:        {scored['f1'].mean():.4f}"
          f"  (in-distribution {scored['f1_in_distribution'].mean():.4f})")
    print(f"  macro-F1 over the {len(survived)} keeping supervision:  {survived['f1'].mean():.4f}"
          f"  (in-distribution {survived['f1_in_distribution'].mean():.4f})")
    if len(starved):
        print(f"  excluded, no supervision left after the holdout ({len(starved)}): "
              f"{', '.join(starved['topic_id'])}")
    collapsed = survived[(survived["f1"] == 0) & (survived["train_positives_remaining"] >= 500)]
    if len(collapsed):
        print("  register transfer failures, F1 0.000 with >=500 positives retained: "
              f"{', '.join(collapsed['topic_id'])}")